<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_05_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.2 · Notebook 05 — assemble the report

**Paired with L10.2 · Solid oxide cells**

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · Collect the cases

Every earlier notebook saved its cases into `Ex10.2_outputs`. This cell loads
whichever of them exist.

In [ ]:
import pickle

OUT = "Ex10.2_outputs"
os.makedirs(OUT, exist_ok=True)

cases = []
for f in ("ex102_fit.pkl", "ex102_cases.pkl", "ex102_opt.pkl"):
    path = os.path.join(OUT, f)
    if os.path.exists(path):
        with open(path, "rb") as fh:
            d = pickle.load(fh)
        cases.extend(d if isinstance(d, list) else [d])
print(f"{len(cases)} cases loaded")

## 2 · Write it

In [ ]:
path = pb.make_report(cases, filename=os.path.join(OUT, "Ex10.2_report.md"),
                      author="YOUR NAME",
                      notes="Replace with anything you want recorded.")
print(open(path).read()[:1600])

### The report as a PDF

Moodle shows a PDF inline and a `.md` only as a download, so the cell below
converts the report you just wrote into a PDF (with the figure, if one was
saved) and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex10.2_report.md into Ex10.2_report.pdf, with any figure
# saved as Ex10.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md = open("Ex10.2_report.md", encoding="utf-8").read()
figs = sorted(glob.glob("Ex10.2_report*.png"))
if figs:
    imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
    i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
    md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

html = markdown.markdown(md, extensions=["fenced_code", "tables"])
css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                       f"<body>{html}</body></html>", base_url=".").write_pdf("Ex10.2_report.pdf")
print("written Ex10.2_report.pdf", f"with {len(figs)} figure(s)" if figs else "")
try:
    from google.colab import files
    files.download("Ex10.2_report.pdf")
except ImportError:
    pass


## Submitting

Question 5 asks which of your numbers rest on `ESTIMATED` parameters. Answer it
precisely — list them. This exercise has no PyBaMM behind it, and knowing which
of your results are defensible is the point.

## Where this leads

The **L13 MiniProject** raises the complexity: a real stack geometry, a
measured degradation data set, coupled SOFC/SOEC cycling, or model-predictive
control against a real price series. See `MiniProject_Turbulence.md` for the
format; a solid oxide track follows the same structure.